In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [2]:

corpus = [
    "natural language processing is amazing",
    "transformers are powerful deep learning models",
    "chatbots can understand human language",
    "python is widely used in ai projects",
    "students enjoy learning machine learning",
    "computer vision works with images and videos",
    "data analysis helps businesses make decisions",
    "neural networks improve with training data",
    "artificial intelligence is changing the world",
    "deep learning models require large datasets"
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)


Number of sentences: 10

Sample corpus:
- natural language processing is amazing
- transformers are powerful deep learning models
- chatbots can understand human language
- python is widely used in ai projects
- students enjoy learning machine learning
- computer vision works with images and videos
- data analysis helps businesses make decisions
- neural networks improve with training data
- artificial intelligence is changing the world
- deep learning models require large datasets


In [3]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 because indexing starts from 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)


Vocabulary size: 50

Word index:
{'learning': 1, 'is': 2, 'language': 3, 'deep': 4, 'models': 5, 'with': 6, 'data': 7, 'natural': 8, 'processing': 9, 'amazing': 10, 'transformers': 11, 'are': 12, 'powerful': 13, 'chatbots': 14, 'can': 15, 'understand': 16, 'human': 17, 'python': 18, 'widely': 19, 'used': 20, 'in': 21, 'ai': 22, 'projects': 23, 'students': 24, 'enjoy': 25, 'machine': 26, 'computer': 27, 'vision': 28, 'works': 29, 'images': 30, 'and': 31, 'videos': 32, 'analysis': 33, 'helps': 34, 'businesses': 35, 'make': 36, 'decisions': 37, 'neural': 38, 'networks': 39, 'improve': 40, 'training': 41, 'artificial': 42, 'intelligence': 43, 'changing': 44, 'the': 45, 'world': 46, 'require': 47, 'large': 48, 'datasets': 49}


In [4]:

input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")
for seq in input_sequences[:10]:
    print(seq)


Total training sequences: 49

Some sequences before padding:
[8, 3]
[8, 3, 9]
[8, 3, 9, 2]
[8, 3, 9, 2, 10]
[11, 12]
[11, 12, 13]
[11, 12, 13, 4]
[11, 12, 13, 4, 1]
[11, 12, 13, 4, 1, 5]
[14, 15]


In [5]:

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])


Maximum sequence length: 7

Padded sequences:
[[ 0  0  0  0  0  8  3]
 [ 0  0  0  0  8  3  9]
 [ 0  0  0  8  3  9  2]
 [ 0  0  8  3  9  2 10]
 [ 0  0  0  0  0 11 12]
 [ 0  0  0  0 11 12 13]
 [ 0  0  0 11 12 13  4]
 [ 0  0 11 12 13  4  1]
 [ 0 11 12 13  4  1  5]
 [ 0  0  0  0  0 14 15]]


In [6]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (49, 6)
Shape of y: (49, 50)


In [7]:

model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:

history = model.fit(X, y, epochs=200, verbose=1)


Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.0000e+00 - loss: 3.9125
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0816 - loss: 3.9085
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1020 - loss: 3.9050
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1020 - loss: 3.9019
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0816 - loss: 3.8981
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0816 - loss: 3.8930
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0816 - loss: 3.8877
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0816 - loss: 3.8803
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0816 - loss: 3.8730 
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0816 - loss: 3.8625
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.0816 - loss: 3.8490
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0816

In [9]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text


In [10]:

print(generate_text("deep learning", 2))
print(generate_text("machine learning", 2))
print(generate_text("python is", 2))


deep learning models require
machine learning learning models
python is widely used


In [11]:

print(generate_text("artificial intelligence", 2))
print(generate_text("data science", 2))
print(generate_text("lstm is", 3))


artificial intelligence is changing
data science analysis helps
lstm is analysis helps businesses
